In [37]:
import pandas as pd
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types
from pyspark.sql.functions import col

In [2]:
pyspark.__file__

'/home/daniel/anaconda3/lib/python3.13/site-packages/pyspark/__init__.py'

In [3]:
spark = SparkSession.builder \
        .master('local[*]') \
        .appName('taxi_data_consolidated') \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 05:22:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
pwd

'/home/daniel/notebooks'

In [5]:
df_green = \
    spark.read \
    .option('header', 'true') \
    .option('recursiveFileLookup', 'true') \
    .parquet('/home/daniel/notebooks/taxi_data_pq/green/')


In [6]:
df_yellow = \
    spark.read \
    .option('header', 'true') \
    .option('recursiveFileLookup', 'true') \
    .parquet('/home/daniel/notebooks/taxi_data_pq/yellow/')

In [9]:
set(df_green.columns) | set(df_yellow.columns)

{'DOLocationID',
 'PULocationID',
 'RatecodeID',
 'VendorID',
 'congestion_surcharge',
 'ehail_fee',
 'extra',
 'fare_amount',
 'improvement_surcharge',
 'lpep_dropoff_datetime',
 'lpep_pickup_datetime',
 'mta_tax',
 'passenger_count',
 'payment_type',
 'store_and_fwd_flag',
 'tip_amount',
 'tolls_amount',
 'total_amount',
 'tpep_dropoff_datetime',
 'tpep_pickup_datetime',
 'trip_distance',
 'trip_type'}

In [10]:
# normalizing green columns: 
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime') \
    .withColumn('taxi_type', F.lit('green')) \
    .select(
        'VendorID', 
        'pickup_datetime', 
        'dropoff_datetime', 
        'store_and_fwd_flag',
        'RatecodeID',
        'PULocationID',
        'DOLocationID',
        'passenger_count',
        'trip_distance',
        'fare_amount',
        'extra',
        'mta_tax',
        'tip_amount',
        'tolls_amount',
        'ehail_fee',
        'improvement_surcharge',
        'total_amount',
        'payment_type',
        'trip_type',
        'congestion_surcharge',
        'taxi_type'
    )

In [11]:
# normalizing yellow columns: 
df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime') \
    .withColumn('taxi_type', F.lit('yellow')) \
    .select(
        'VendorID', 
        'pickup_datetime', 
        'dropoff_datetime', 
        'passenger_count',
        'trip_distance',
        'RatecodeID',
        'store_and_fwd_flag',
        'PULocationID',
        'DOLocationID',
        'payment_type',
        'fare_amount',
        'extra',
        'mta_tax',
        'tip_amount',
        'tolls_amount',
        'improvement_surcharge',
        'total_amount',
        'congestion_surcharge', 
        'taxi_type'
    )

In [12]:
commons_cols = []

for col_green in df_green.columns:
    for col_yellow in df_yellow.columns:
        
        if col_green == col_yellow: 
            commons_cols.append(col_green)
        
        else: 
            None

In [13]:
commons_cols

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'congestion_surcharge',
 'taxi_type']

In [14]:
df_green = df_green.select(commons_cols)

In [15]:
df_green.select(F.count('*')).show()

[Stage 2:=======================================>                   (4 + 2) / 6]

+--------+
|count(1)|
+--------+
| 8348567|
+--------+



In [16]:
df_yellow = df_yellow.select(commons_cols)

In [17]:
df_yellow.select(F.count('*')).show()

[Stage 5:==================================================>      (30 + 4) / 34]

+---------+
| count(1)|
+---------+
|124048218|
+---------+



In [18]:
yellow_green_dataset_unioned = df_yellow.union(df_green)

In [19]:
yellow_green_dataset_unioned.select(F.count('*')).show()

[Stage 8:====================================================>    (37 + 3) / 40]

+---------+
| count(1)|
+---------+
|132396785|
+---------+



In [20]:
yellow_green_dataset_unioned \
    .createOrReplaceTempView('yellow_green_dataset')

In [21]:
#Sql Notation: 
df_yellow_green_dataset = spark.sql(
    """
    SELECT
        -- Reveneue grouping
        PULocationID AS revenue_zone_ID,
        date_trunc('month', pickup_datetime) AS revenue_month,
        taxi_type,
    
        -- Revenue calculation
        SUM(fare_amount) AS revenue_monthly_fare,
        SUM(extra) AS revenue_monthly_extra,
        SUM(mta_tax) AS revenue_monthly_mta_tax,
        SUM(tip_amount) AS revenue_monthly_tip_amount,
        SUM(tolls_amount) AS revenue_monthly_tolls_amount,
        SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
        SUM(total_amount) AS revenue_monthly_total_amount,
        SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,
    
        -- Additional calculations
        -- COUNT(tripid) AS total_monthly_trips,
        AVG(passenger_count) AS avg_montly_passenger_count,
        AVG(trip_distance) AS avg_montly_trip_distance

    FROM yellow_green_dataset
    
    GROUP BY
        PULocationID,
        date_trunc('month', pickup_datetime),
        taxi_type
    ORDER BY 
        revenue_monthly_total_amount DESC
    """
)

In [22]:
df_yellow_green_dataset.show(5)

[Stage 11:===================================================>    (37 + 3) / 40]

+---------------+-------------------+---------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+--------------------------+------------------------+
|revenue_zone_ID|      revenue_month|taxi_type|revenue_monthly_fare|revenue_monthly_extra|revenue_monthly_mta_tax|revenue_monthly_tip_amount|revenue_monthly_tolls_amount|revenue_monthly_improvement_surcharge|revenue_monthly_total_amount|revenue_monthly_congestion_surcharge|avg_montly_passenger_count|avg_montly_trip_distance|
+---------------+-------------------+---------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+--------------------------+------------------------+
|            132|20

In [56]:
#SparkSql Notation: 

yellow_green_dataset =  \
    yellow_green_dataset_unioned \
        .withColumn('revenue_month', F.date_trunc('month', 'pickup_datetime')) \
        .select(
            #Grouping columns
            yellow_green_dataset_unioned.PULocationID.alias('revenue_zone_ID'), 
            'revenue_month', 
            yellow_green_dataset_unioned.taxi_type,
            
            #Aggregations
            yellow_green_dataset_unioned.fare_amount.alias('revenue_monthly_fare'),
            yellow_green_dataset_unioned.extra.alias('revenue_monthly_extra'),
            yellow_green_dataset_unioned.mta_tax.alias('revenue_monthly_mta_tax'),
            yellow_green_dataset_unioned.tip_amount.alias('revenue_monthly_tip_amount'),
            yellow_green_dataset_unioned.tolls_amount.alias('revenue_monthly_tolls_amount'),
            yellow_green_dataset_unioned.improvement_surcharge.alias('revenue_monthly_improvement_surcharge'),
            yellow_green_dataset_unioned.total_amount.alias('revenue_monthly_total_amount'),
            yellow_green_dataset_unioned.congestion_surcharge.alias('revenue_monthly_congestion_surcharge'),

            #Additional agregations: 
            yellow_green_dataset_unioned.passenger_count.alias('avg_montly_passenger_count'), 
            yellow_green_dataset_unioned.trip_distance.alias('avg_montly_trip_distance') 
        ) \
        .groupBy(
            'revenue_zone_ID',
            'revenue_month',
            yellow_green_dataset_unioned.taxi_type
        ) \
        .agg(
            F.sum('revenue_monthly_fare').alias('revenue_monthly_fare'), 
            F.sum('revenue_monthly_extra').alias('revenue_monthly_extra'),
            F.sum('revenue_monthly_mta_tax').alias('revenue_monthly_mta_tax'),
            F.sum('revenue_monthly_tip_amount').alias('revenue_monthly_tip_amount'),
            F.sum('revenue_monthly_tolls_amount').alias('revenue_monthly_tolls_amount'),
            F.sum('revenue_monthly_improvement_surcharge').alias('revenue_monthly_improvement_surcharge'),
            F.sum('revenue_monthly_total_amount').alias('revenue_monthly_total_amount'),
            F.sum('revenue_monthly_congestion_surcharge').alias('revenue_monthly_congestion_surcharge'),

            F.avg('avg_montly_passenger_count').alias('avg_montly_passenger_count'),
            F.avg('avg_montly_trip_distance').alias('avg_montly_trip_distance')
        ) \
        .orderBy(
            F.desc('revenue_monthly_total_amount')
        )


In [57]:
yellow_green_dataset.show(5)

[Stage 17:=====================================================>  (38 + 2) / 40]

+---------------+-------------------+---------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+--------------------------+------------------------+
|revenue_zone_ID|      revenue_month|taxi_type|revenue_monthly_fare|revenue_monthly_extra|revenue_monthly_mta_tax|revenue_monthly_tip_amount|revenue_monthly_tolls_amount|revenue_monthly_improvement_surcharge|revenue_monthly_total_amount|revenue_monthly_congestion_surcharge|avg_montly_passenger_count|avg_montly_trip_distance|
+---------------+-------------------+---------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+--------------------------+------------------------+
|            132|20

In [61]:
#Export the dataset agregated to pq:

output_file = 'taxi_data_aggregated.parquet'

yellow_green_dataset \
    .write.parquet(f'/home/daniel/notebooks/taxi_data_reporting/{output_file}')

In [73]:
!tree -lh taxi_data_reporting/

[4.0K]  taxi_data_reporting/
└── [4.0K]  taxi_data_aggregated.parquet
    ├── [   0]  _SUCCESS
    └── [865K]  part-00000-2c9d3211-fc9c-4c99-bd06-b40e1bb951a5-c000.snappy.parquet

1 directory, 2 files


In [71]:
!wc -l taxi_data_reporting/taxi_data_aggregated.parquet/part-00000-2c9d3211-fc9c-4c99-bd06-b40e1bb951a5-c000.snappy.parquet

6318 taxi_data_reporting/taxi_data_aggregated.parquet/part-00000-2c9d3211-fc9c-4c99-bd06-b40e1bb951a5-c000.snappy.parquet
